# Daily Reload: lifetime wipe-and-refetch, or targeted date-range (thin notebook)

**Git holds only the reusable functions** (the `dhan-pipeline` package).
**This notebook holds every variable** — mode, dates, buffer, table names.
Edit the VARIABLES cell, then run.

**Two modes:**
- `lifetime` — deletes the ENTIRE daily table, refetches every scrip from
  `LIFETIME_START` to today, reloads. No split check needed: a full refetch
  always returns Dhan's current, already-adjusted prices.
- `range` — deletes only `[FROM_DATE, TO_DATE]`, refetches that window (plus
  `BUFFER_DAYS` of lookback), and checks the OLDEST date in the range against
  what's currently stored in BigQuery for that date. Any scrip whose price
  moved gets its ENTIRE lifetime history deleted + refetched (its whole
  series is stale post-split); everyone else just gets the range reuploaded.

⚠️ **Destructive**: this DELETES data before reuploading. Double-check MODE
and the dates before running, especially `lifetime`.

In [ ]:
# 1. Install the shared functions from GitHub (fast: skips deps Colab already has)
!pip install -q --force-reinstall --no-deps "git+https://github.com/rajatjain1992/dhan-pipeline.git"

In [ ]:
# 2. Auth: BigQuery via Colab, Drive for the service-account JSON (needed for gspread)
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

In [ ]:
# 3. ===== VARIABLES — edit these, this is the only place values live =====
from dhan_pipeline import Config, run_daily_reload
from dhan_pipeline import load_scrip_mapping, gspread_client

MODE = 'range'          # 'lifetime' or 'range'

# --- only used when MODE == 'range' ---
FROM_DATE    = '2026-07-01'
TO_DATE      = '2026-08-08'
BUFFER_DAYS  = 5         # extra calendar days fetched BEFORE FROM_DATE as lookback
                          # context. REMINDER: raise this if FROM_DATE sits right
                          # after a weekend/holiday gap, so the fetch window
                          # comfortably contains real trading days.

cfg = Config(
    dhan_client_id    = 'PASTE_CLIENT_ID',
    dhan_access_token = 'PASTE_FRESH_DHAN_TOKEN',
    service_account_file = '/content/drive/MyDrive/Colab Notebooks/rajat-trade-c411eaec7c51.json',
    project_id    = 'rajat-trade',
    dataset_id    = 'stock_data_set',
    daily_table   = 'stock_daily_prices_dhan',
    staging_table = 'stock_daily_prices_dhan_staging',
    flag_table    = 'corporate_action_flags',
    sheet_key          = '1aoEgOhQkAAv8b2NqAWtZUYXG41rOal77i0XasevyNtE',
    list_worksheet     = 'my_list',
    negative_worksheet = 'Negative List',
)

In [ ]:
# 3b. Load the scrip list from the Google Sheet.
gc = gspread_client(cfg)
scrip_mapping = load_scrip_mapping(cfg, gc)
print(f'Total scrips: {len(scrip_mapping)}')

In [ ]:
# 4. Run.
if MODE == 'lifetime':
    result = run_daily_reload(cfg, scrip_mapping, mode='lifetime')
else:
    result = run_daily_reload(
        cfg, scrip_mapping, mode='range',
        from_date=FROM_DATE, to_date=TO_DATE, buffer_days=BUFFER_DAYS,
    )
result